In [ ]:
import pandas as pd
import sqlite3

# Load CSV
df = pd.read_csv("/content/engineered_dataset.csv")

# Check first few rows
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,...,Previous Purchases Normalized,Frequency Normalized,Customer Value Score,Satisfaction Score,Subscription Status Binary,Behavioral Loyalty Score,Attitudinal Loyalty Score,Loyalty Score,Value Tier,Satisfaction Flag
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,0.265306,0.6,0.462092,0.62,1,0.431286,0.72,0.431286,Low,0
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,0.020408,0.6,0.416122,0.62,1,0.277415,0.72,0.277415,Low,0
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,0.448980,0.8,0.667194,0.62,1,0.639394,0.72,0.639394,Medium,0
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,0.979592,0.8,0.868878,0.70,1,0.851500,0.80,0.851500,High,0
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,0.612245,0.0,0.256173,0.54,1,0.248168,0.64,0.248168,Low,0


In [ ]:
# Create SQLite connection
conn = sqlite3.connect("my_database.db")

# Save dataframe as SQL table
df.to_sql("engineered_dataset", conn, if_exists="replace", index=False)

3900

In [ ]:
df['Frequency Score'].head()

,Frequency Score
0,4
1,4
2,5
3,5
4,1


In [ ]:
query = """
SELECT "Value Tier",
COUNT(*) AS "Value Tier Count"
FROM engineered_dataset
GROUP BY "Value Tier";
"""

pd.read_sql_query(query, conn)

,Value Tier,Value Tier Count
0,High,254
1,Low,2219
2,Medium,1427


In [ ]:
query = """
SELECT
    "Value Tier",
    AVG("Loyalty Score") AS "Average Loyalty Score",
    AVG("Subscription Status Binary") AS "Average Subscription Rate",
    AVG("Promo Dependency") AS "Average Promo Dependency"
FROM engineered_dataset
GROUP BY "Value Tier";
"""

pd.read_sql_query(query, conn)

,Value Tier,Average Loyalty Score,Average Subscription Rate,Average Promo Dependency
0,High,0.844619,0.287402,0.011667
1,Low,0.320018,0.264984,0.036137
2,Medium,0.612927,0.274702,0.023469


In [ ]:
query = """
SELECT
    "Value Tier",
    AVG("Loyalty Score") AS "Average Loyalty Score",
    AVG("Subscription Status Binary") AS "Average Subscription Rate",
    AVG("Promo Dependency") AS "Average Promo Dependency",
    AVG("Frequency Score") AS "Average Frequency of Purchases"
FROM engineered_dataset
GROUP BY "Value Tier";
"""

pd.read_sql_query(query, conn)


,Value Tier,Average Loyalty Score,Average Subscription Rate,Average Promo Dependency,Average Frequency of Purchases
0,High,0.844619,0.287402,0.011667,5.775591
1,Low,0.320018,0.264984,0.036137,2.153673
2,Medium,0.612927,0.274702,0.023469,4.501752


**Query 1 Finding: What separates high-value customers from low-value ones?**

**Promo Dependency** inversely correlates with value tier — high-value customers have an average dependency score of 0.01 vs 0.036 for low-value customers, indicating they buy without needing discounts.

**Purchase Frequency** is 2.5x higher in the high-value tier (5.77) compared to low-value customers (2.15), signaling strong repeat purchase behavior.

**Loyalty Score** follows the same pattern — high-value customers score 0.84 on average vs 0.32 for low-value, reflecting sustained engagement with the brand.

**Subscription Rate** is slightly higher among high-value customers (28.7%) vs low-value (26.5%), though the difference is less pronounced.

**Takeaway**: High-value customers are organically engaged and brand-loyal, while low-value customers appear primarily discount-driven and transactional.


In [ ]:
query = """
SELECT AVG("PREVIOUS PURCHASES"), "SEASON", "CATEGORY"
FROM engineered_dataset
GROUP BY "SEASON", "CATEGORY";
"""

pd.read_sql_query(query, conn)


,"AVG(""PREVIOUS PURCHASES"")",Season,Category
0,24.651235,Fall,Accessories
1,25.454333,Fall,Clothing
2,24.588235,Fall,Footwear
3,24.386364,Fall,Outerwear
4,25.943522,Spring,Accessories
5,24.759912,Spring,Clothing
6,24.165644,Spring,Footwear
7,25.950617,Spring,Outerwear
8,26.804487,Summer,Accessories
9,24.460784,Summer,Clothing


In [ ]:
query = """
SELECT
    CASE
        WHEN "Previous Purchases" > 24 THEN 'High Tenure'
        ELSE 'Low Tenure'
    END AS "Tenure Type",
    "Season",
    "Category",
    COUNT(*) AS "Customer Count"
FROM engineered_dataset
GROUP BY "Tenure Type", "Season", "Category"
ORDER BY "Season", "Category", COUNT(*) DESC;
"""

pd.read_sql_query(query, conn)

,Tenure Type,Season,Category,Customer Count
0,Low Tenure,Fall,Accessories,172
1,High Tenure,Fall,Accessories,152
2,High Tenure,Fall,Clothing,219
3,Low Tenure,Fall,Clothing,208
4,Low Tenure,Fall,Footwear,70
5,High Tenure,Fall,Footwear,66
6,High Tenure,Fall,Outerwear,46
7,Low Tenure,Fall,Outerwear,42
8,High Tenure,Spring,Accessories,158
9,Low Tenure,Spring,Accessories,143


**Query 2 Finding: Which seasons and categories are associated with lower-tenure vs higher-tenure customers?**

**Clothing:** Consistently over-indexes among high-tenure customers across all seasons — the brand's strongest retention category.
Accessories: Leans toward high-tenure customers in 3 out of 4 seasons, suggesting it also functions as a retention driver.

**Footwear:** Mixed pattern across seasons — neither clearly retention nor entry-point, suggesting it attracts both new and returning customers equally.
Outerwear: No meaningful tenure difference — likely a seasonal necessity purchase rather than a loyalty signal.

**Summer, Winter, Fall:** High-tenure customers dominate in 3 out of 4 categories — these seasons drive repeat purchases from established customers.
Spring: Most balanced season between tenure groups — likely the strongest window for new customer acquisition.

**Takeaway:** Clothing and Accessories are the brand's retention anchors, while Spring represents the clearest acquisition opportunity. The brand should prioritize new customer campaigns in Spring and use Clothing and Accessories as loyalty levers year-round.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 32 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Customer ID                    3900 non-null   int64  
 1   Age                            3900 non-null   int64  
 2   Gender                         3900 non-null   object 
 3   Item Purchased                 3900 non-null   object 
 4   Category                       3900 non-null   object 
 5   Purchase Amount (USD)          3900 non-null   int64  
 6   Location                       3900 non-null   object 
 7   Size                           3900 non-null   object 
 8   Color                          3900 non-null   object 
 9   Season                         3900 non-null   object 
 10  Review Rating                  3900 non-null   float64
 11  Subscription Status            3900 non-null   object 
 12  Shipping Type                  3900 non-null   o

In [ ]:
query = """
SELECT
    CASE
        WHEN AVG("Promo Dependency") > (
            SELECT AVG("Promo Dependency")
            FROM engineered_dataset
        )
        THEN 'High Dependency'
        ELSE 'Low Dependency'
    END AS "Dependency Type",
    "Location",
    AVG("Promo Dependency") AS "Average Promo Dependency",
    COUNT(*) AS "Customer Count"
FROM engineered_dataset
GROUP BY "Location";
"""

pd.read_sql_query(query, conn)

,Dependency Type,Location,Average Promo Dependency,Customer Count
0,High Dependency,Alabama,0.033224,89
1,Low Dependency,Alaska,0.019580,72
2,Low Dependency,Arizona,0.021118,65
3,Low Dependency,Arkansas,0.028629,79
4,Low Dependency,California,0.026553,95
5,High Dependency,Colorado,0.035551,75
6,Low Dependency,Connecticut,0.023802,78
7,High Dependency,Delaware,0.045495,86
8,Low Dependency,Florida,0.023341,68
9,High Dependency,Georgia,0.036072,79


In [ ]:
query = """
SELECT "Dependency Type", COUNT(*) AS "Location Count"
FROM (
    SELECT
        CASE
            WHEN AVG("Promo Dependency") > (
                SELECT AVG("Promo Dependency")
                FROM engineered_dataset
            )
            THEN 'High Dependency'
            ELSE 'Low Dependency'
        END AS "Dependency Type",
        "Location",
        AVG("Promo Dependency") AS "Average Promo Dependency",
        COUNT(*) AS "Customer Count"
    FROM engineered_dataset
    GROUP BY "Location"
) AS subquery
GROUP BY "Dependency Type";
"""

pd.read_sql_query(query, conn)

,Dependency Type,Location Count
0,High Dependency,21
1,Low Dependency,29


**Query 3 Finding: Which geographies signal organic demand versus discount-driven volume?**

**Threshold used:** Overall average promo dependency across all locations, used as a relative benchmark to classify states as above or below average.

**High Dependency states (21 states):** Including Alabama, Georgia, Indiana, and Nevada — these states show above-average reliance on discounts to drive purchases. Reducing promos here risks a drop in conversion and volume.

**Low Dependency states (29 states):** Including Alaska, Hawaii, Illinois, and Kansas — these states demonstrate organic brand pull, with customers buying independently of promotional activity. The brand has genuine traction here.

**Business implication for High Dependency states:** Promos are likely a purchase trigger, not just a nice-to-have. Any discount reduction should be gradual and closely monitored.

**Business implication for Low Dependency states:** Safe to reduce or eliminate promos without significant volume risk — these markets represent the brand's strongest margin opportunity.

**Takeaway:** 29 out of 50 states show organic demand, meaning the brand has genuine pull in the majority of its markets. The 21 high-dependency states require a more careful, phased approach to any promotional restructuring.